# 4.2 Primary outcome - heterogeneity analysis

This notebook does the following:
    - Simple diff in diff analysis of primary outcome (energy use) - heterogeneity by: (i) faculty, (ii) BL energy consumption, (iii) lab group size

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
import importlib
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import os
import statsmodels.formula.api as smf
import pyfixest as pf
from make_regression_table import make_regression_table

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Construct post variable
df["post"] = (df["survey"] == "EL").astype(int)

In [4]:
# Keep only labgroups that have both pre and post observations
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

## (1) Simple diff in diff regression


In [5]:
fit_levels = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + post",
    data=df,
    vcov={"CRV1": "labgroupid"}
)

fit_levels.summary()

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


###

Estimation:  OLS
Dep. var.: annual_electricity_total, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  218

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |     2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|---------:|--------:|
| treated:post  |    -19.563 |      111.782 |    -0.175 |      0.861 | -241.136 | 202.009 |
---
RMSE: 288.413 R2: 1.0 R2 Within: 0.0 


In [6]:
df['log_electricity'] = np.log1p(df['annual_electricity_total']) 

fit_log = pf.feols(
    "log_electricity ~ treated:post | labgroupid + post",
    data=df,
    vcov={"CRV1": "labgroupid"}
)
fit_log.summary()

###

Estimation:  OLS
Dep. var.: log_electricity, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  218

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treated:post  |     -0.053 |        0.051 |    -1.038 |      0.302 | -0.153 |   0.048 |
---
RMSE: 0.127 R2: 0.994 R2 Within: 0.011 


## (2) Heterogeneity analysis

In [7]:
# Define subsamples for faculty
df_mnf  = df[df["faculty"] == "Faculty of Science (MNF)"].copy()
df_mef  = df[df["faculty"] == "Faculty of Medicine (MeF)"].copy()
df_both = df[df["faculty"] == "Both MNF and MeF"].copy()

# Define subsamples for baseline energy use (below/above median)
bl = df[df["survey"] == "BL"]
median_energy = bl["annual_electricity_total"].median()
low_bl_energy_labs  = bl[bl["annual_electricity_total"] <= median_energy]["labgroupid"].unique()
high_bl_energy_labs = bl[bl["annual_electricity_total"] >  median_energy]["labgroupid"].unique()
df_low_bl  = df[df["labgroupid"].isin(low_bl_energy_labs)].copy()
df_high_bl = df[df["labgroupid"].isin(high_bl_energy_labs)].copy()

# Alternative spec: IT equipment only vs. also other equipment
it_only_labs       = bl[bl["annual_electricity_it"] == bl["annual_electricity_total"]]["labgroupid"].unique()
other_equipment_labs = bl[bl["annual_electricity_it"] != bl["annual_electricity_total"]]["labgroupid"].unique()
df_it_only        = df[df["labgroupid"].isin(it_only_labs)].copy()
df_other_equipment = df[df["labgroupid"].isin(other_equipment_labs)].copy()

# Define subsamples for lab group size
median_size = df["no_researchers"].median()
df_small_labs = df[df["no_researchers"] <= median_size].copy()
df_large_labs = df[df["no_researchers"] >  median_size].copy()

In [8]:
# Heterogeneity analyses — store results by subsample name
subsamples = {
    "MNF":               df_mnf,
    "MeF":               df_mef,
    "Both":              df_both,
    "Low BL Energy":     df_low_bl,
    "High BL Energy":    df_high_bl,
    "IT Only":           df_it_only,
    "Other Equipment":   df_other_equipment,
    "Small Labs":        df_small_labs,
    "Large Labs":        df_large_labs,
}

fit_levels_sub = {}
fit_log_sub    = {}

for name, df_sub in subsamples.items():
    fit_levels_sub[name] = pf.feols(
        "annual_electricity_total ~ treated:post | labgroupid + post",
        data=df_sub, vcov={"CRV1": "labgroupid"}
    )
    fit_log_sub[name] = pf.feols(
        "log_electricity ~ treated:post | labgroupid + post",
        data=df_sub, vcov={"CRV1": "labgroupid"}
    )

In [9]:
# Export to nice table
table = make_regression_table(
    fit_list = [
        fit_levels,
        fit_levels_sub["MNF"], fit_levels_sub["MeF"], fit_levels_sub["Both"],
        fit_levels_sub["Low BL Energy"], fit_levels_sub["High BL Energy"],
        fit_levels_sub["IT Only"], fit_levels_sub["Other Equipment"],
        fit_levels_sub["Small Labs"], fit_levels_sub["Large Labs"],
    ],
    model_names   = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)", "(8)", "(9)", "(10)"],
    keep_vars     = ["treated:post"],
    var_labels    = {"treated:post": "Treated $\\times$ Post"},
    fe_rows       = {
        "Lab group FE": [True] * 10,
        "Time FE":      [True] * 10,
    },
    col_groups = {
        "All": [0],
        "Faculty": [1,2,3],
        "Baseline Energy Use": [4,5],
        "Wet or Dry": [6,7],
        "Group Size": [8,9],
    },
    col_subgroups = {
        "Science": [1],
        "Medicine": [2],
        "Joint": [3],
        "Below median": [4,8],
        "Above median": [5,9],
        "Dry": [6],
        "Wet": [7],
    },
    baseline_mean  = "auto",
    decimals       = [1] * 10,
    mean_decimals = 0,
    r2_type        = None,
    col1_width     = "4cm",
    coln_width     = "1.5cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "simple_diff_in_diff_het_levels.tex"
_ = table_path.write_text(table)

In [10]:
# Export to nice table
table = make_regression_table(
    fit_list = [
        fit_log,
        fit_log_sub["MNF"], fit_log_sub["MeF"], fit_log_sub["Both"],
        fit_log_sub["Low BL Energy"], fit_log_sub["High BL Energy"],
        fit_log_sub["IT Only"], fit_log_sub["Other Equipment"],
        fit_log_sub["Small Labs"], fit_log_sub["Large Labs"],
    ],
    model_names   = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)", "(8)", "(9)", "(10)"],
    keep_vars     = ["treated:post"],
    var_labels    = {"treated:post": "Treated $\\times$ Post"},
    fe_rows       = {
        "Lab group FE": [True] * 10,
        "Time FE":      [True] * 10,
    },
    col_groups = {
        "All": [0],
        "Faculty": [1,2,3],
        "Baseline Energy Use": [4,5],
        "Wet or Dry": [6,7],
        "Group Size": [8,9],
    },
    col_subgroups = {
        "Science": [1],
        "Medicine": [2],
        "Joint": [3],
        "Below median": [4,8],
        "Above median": [5,9],
        "Dry": [6],
        "Wet": [7],
    },
    baseline_mean  = "auto",
    decimals       = [3] * 10,
    mean_decimals = 3,
    r2_type        = None,
    col1_width     = "4cm",
    coln_width     = "1.5cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "simple_diff_in_diff_het_log.tex"
_ = table_path.write_text(table)